# LeetCode 3545 — Minimum Deletions for At Most K Distinct Characters bằng Python

Link đề: https://leetcode.com/problems/minimum-deletions-for-at-most-k-distinct-characters/ · Độ khó: **Easy** · Chủ đề: String, Hash Table, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def minDeletion(self, s: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được **xoá** một số ký tự (có thể không xoá) khỏi chuỗi `s` sao cho còn **tối đa `k` ký tự khác nhau**; tìm **số ký tự phải xoá ít nhất**.
2. Đếm tần suất ký tự bằng `collections.Counter` (hoặc mảng 26 ô) và hiểu rằng khi xoá một **chữ cái**, ta xoá **toàn bộ** các lần xuất hiện của nó.
3. Rút ra tham lam: **giữ lại `k` chữ cái xuất hiện nhiều nhất**, xoá tất cả chữ cái còn lại (tổng số lần của chúng).
4. Viết 4 cách: vét cạn mọi tập chữ cái giữ lại (mặt nạ bit), sắp xếp tần suất tăng dần, `most_common(k)`, và min-heap; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: xoá chữ cái **nhiều** nhất thay vì ít nhất, đếm số **chữ cái** thay vì số **ký tự** bị xoá, quên trường hợp đã có `<= k` chữ cái, và xoá từng ký tự lẻ tẻ.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ cái thường và số nguyên `k`. Bạn được **xoá** một số ký tự của `s` (có thể không xoá ký tự nào). Hãy tìm **số lượng ký tự tối thiểu cần xoá** sao cho chuỗi còn lại có **không quá `k` ký tự khác nhau** (distinct).

| Ví dụ | s, k | Output | Giải thích |
|---|---|---|---|
| 1 | `"abc"`, `2` | 1 | có 3 chữ cái khác nhau, cần còn tối đa 2: xoá một chữ cái (ví dụ `c`) -> xoá 1 ký tự |
| 2 | `"aabb"`, `2` | 0 | đã chỉ có 2 chữ cái khác nhau |
| 3 | `"yyyzz"`, `1` | 2 | giữ `y` (xuất hiện 3 lần), xoá toàn bộ `z` (2 ký tự) |

Constraints: `1 <= s.length <= 16`; `1 <= k <= 16`; `s` chỉ gồm chữ cái thường `a-z`.

In [1]:
from collections import Counter

s, k = "yyyzz", 1
cnt = Counter(s)
print("tần suất:", dict(cnt), "| số chữ cái khác nhau =", len(cnt))
freqs = sorted(cnt.values())
need_remove = max(0, len(cnt) - k)
print("cần loại", need_remove, "chữ cái ít nhất -> xoá", sum(freqs[:need_remove]), "ký tự")

tần suất: {'y': 3, 'z': 2} | số chữ cái khác nhau = 2
cần loại 1 chữ cái ít nhất -> xoá 2 ký tự


## 3. Kiến thức nền

### 3.1. Ký tự "khác nhau" (distinct) và `set`

"Số ký tự khác nhau" của một chuỗi là số **chữ cái phân biệt** trong chuỗi: `len(set(s))`. Chuỗi `"aabb"` có `2` chữ cái khác nhau (`a`, `b`) dù dài `4`. Ràng buộc "tối đa `k` ký tự khác nhau" nghĩa là `len(set(chuỗi_còn_lại)) <= k`.

In [2]:
for s in ("aabb", "abc", "yyyzz", "a"):
    print(repr(s), "dài", len(s), "| khác nhau:", len(set(s)))

'aabb' dài 4 | khác nhau: 2
'abc' dài 3 | khác nhau: 3
'yyyzz' dài 5 | khác nhau: 2
'a' dài 1 | khác nhau: 1


### 3.2. `collections.Counter` — đếm tần suất

`Counter(s)` đếm số lần xuất hiện của từng ký tự, giống một dict `ký_tự -> số_lần`. `cnt.values()` cho các tần suất; `len(cnt)` là số chữ cái khác nhau; `cnt.most_common(k)` trả về `k` cặp `(ký_tự, số_lần)` **nhiều nhất**, sắp xếp giảm dần.

In [3]:
from collections import Counter
cnt = Counter("abcaab")
print(cnt, len(cnt), sorted(cnt.values()))
print(cnt.most_common(2))

Counter({'a': 3, 'b': 2, 'c': 1}) 3 [1, 2, 3]
[('a', 3), ('b', 2)]


### 3.3. Xoá một chữ cái = xoá tất cả lần xuất hiện của nó

Để loại một chữ cái khỏi tập "ký tự khác nhau", ta phải xoá **mọi** ký tự bằng nó. Chi phí loại chữ cái `c` là `cnt[c]` (số lần xuất hiện), **không** phải `1`. Xoá chỉ một vài bản của `c` không giảm số chữ cái khác nhau, nên vô ích.

In [4]:
cnt = Counter("yyyzz")
print("loại z tốn", cnt["z"], "ký tự; loại y tốn", cnt["y"], "ký tự")

loại z tốn 2 ký tự; loại y tốn 3 ký tự


### 3.4. Cắt lát danh sách đã sắp xếp: `freqs[:m]`

`freqs = sorted(cnt.values())` (tăng dần). `freqs[:m]` là `m` tần suất **nhỏ nhất**; `sum(freqs[:m])` là tổng chúng. Chú ý: nếu `m` **âm**, `freqs[:m]` cắt từ cuối (`freqs[:-1]` bỏ phần tử cuối) — một cái bẫy; cần đảm bảo `m = max(0, d - k)`.

In [5]:
freqs = [1, 2, 3, 4]
print(freqs[:2], sum(freqs[:2]))
print(freqs[:0], sum(freqs[:0]))
print(freqs[:-1], "<- m âm cắt kiểu khác, nguy hiểm")

[1, 2] 3
[] 0
[1, 2, 3] <- m âm cắt kiểu khác, nguy hiểm


### 3.5. Min-heap `heapq`

`heapq` quản lý hàng đợi ưu tiên: `heapify(list)` biến list thành min-heap; `heappop(h)` lấy phần tử **nhỏ nhất** (`O(log n)`). Hợp với bài này: "liên tục loại chữ cái xuất hiện **ít nhất** cho tới khi còn `k` chữ cái".

In [6]:
import heapq
h = [3, 1, 2]
heapq.heapify(h)
print(heapq.heappop(h), heapq.heappop(h), h)

1 2 [3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với `s = "abcaab"`, `k = 2`: tần suất `a:3`, `b:2`, `c:1` — 3 chữ cái, cần còn tối đa 2, nên **phải loại đúng `3 - 2 = 1` chữ cái**. Loại chữ cái nào cho ít ký tự bị xoá nhất? Loại `c` (chỉ `1` ký tự). Xoá `1` ký tự.

```
giữ {a, b}: xoá c        -> 1 ký tự
giữ {a, c}: xoá b (2)    -> 2 ký tự
giữ {b, c}: xoá a (3)    -> 3 ký tự
```

Nguyên tắc: mỗi chữ cái hoặc **giữ hết** hoặc **xoá hết**; muốn xoá ít nhất thì **giữ những chữ cái xuất hiện nhiều**, xoá những chữ cái xuất hiện ít.

## 5. Cách 1 — Brute Force / baseline

Vì `s.length <= 16`, số chữ cái khác nhau `d <= 16`. Thử **mọi tập con** chữ cái được giữ (mặt nạ bit `2^d`): nếu tập có kích thước `<= k` thì số ký tự được giữ là tổng `cnt` của chúng; chọn tập giữ **nhiều ký tự nhất**. Đáp án = `len(s) - số_ký_tự_giữ_tối_đa`.

Nút thắt: `2^d` tập con -> bùng nổ nếu `d` lớn; với `d <= 16` (65 nghìn tập) vẫn chạy được trong constraints, nhưng không là cách tốt. Dùng làm **chuẩn chân lý** trực tiếp theo định nghĩa.

In [7]:
from collections import Counter

class SolutionBrute:
    def minDeletion(self, s, k):
        cnt = Counter(s)
        letters = list(cnt)
        d = len(letters)
        best_keep = 0
        for mask in range(1 << d):
            if bin(mask).count("1") > k:
                continue
            keep = sum(cnt[letters[i]] for i in range(d) if mask >> i & 1)
            best_keep = max(best_keep, keep)
        return len(s) - best_keep

print(SolutionBrute().minDeletion("abc", 2))
print(SolutionBrute().minDeletion("aabb", 2))
print(SolutionBrute().minDeletion("yyyzz", 1))

1
0
2


**Độ phức tạp:** Time `O(n + 2^d * d)` với `d <= min(n, 26)` (ở đây `d <= 16`), Space `O(d)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** việc chọn tập giữ lại chỉ phụ thuộc **tần suất**. Cần loại đúng `m = max(0, d - k)` chữ cái; chi phí loại một chữ cái là tần suất của nó; để **tổng chi phí nhỏ nhất** ta loại `m` chữ cái có tần suất **nhỏ nhất**. (Tham lam đúng vì chi phí các chữ cái độc lập và cộng được.)

**Quy tắc "nếu… thì…":**

- Nếu `d <= k` -> đã thoả, đáp án `0`.
- Ngược lại -> sắp xếp tần suất tăng dần, đáp án = tổng `d - k` tần suất đầu.

Invariant: sau khi loại `t` chữ cái nhỏ nhất, số chữ cái còn lại là `d - t`, và tổng bị xoá là nhỏ nhất trong mọi cách loại `t` chữ cái.

In [8]:
def greedy(s, k):
    freqs = sorted(Counter(s).values())
    m = max(0, len(freqs) - k)
    return sum(freqs[:m])

print(greedy("abc", 2), greedy("aabb", 2), greedy("yyyzz", 1), greedy("abcaab", 2))

1 0 2 1


## 7. Các cách tiếp theo

Ta có 4 cách: (1) vét cạn tập chữ cái giữ lại; (2) sắp xếp tần suất tăng dần và cộng `d - k` phần tử đầu; (3) `most_common(k)` — giữ `k` chữ cái nhiều nhất; (4) min-heap loại dần chữ cái ít nhất.

### Cách 2 — Sắp xếp tần suất tăng dần

`freqs = sorted(Counter(s).values())`, `m = max(0, d - k)`, trả về `sum(freqs[:m])`.

In [9]:
class SolutionSort:
    def minDeletion(self, s, k):
        freqs = sorted(Counter(s).values())
        m = max(0, len(freqs) - k)
        return sum(freqs[:m])

print(SolutionSort().minDeletion("abc", 2))
print(SolutionSort().minDeletion("aabb", 2))
print(SolutionSort().minDeletion("yyyzz", 1))

1
0
2


Độ phức tạp Cách 2: Time `O(n + d log d)` (`d <= 26`), Space `O(d)`.

### Cách 3 — Giữ `k` chữ cái nhiều nhất (`most_common`)

Góc nhìn ngược: tính số ký tự **giữ lại** = tổng tần suất của `k` chữ cái nhiều nhất (`most_common(k)`), đáp án = `len(s) - giữ`. Nếu `d <= k` thì `most_common(k)` trả toàn bộ nên `giữ = len(s)` và đáp án `0`, không cần nhánh riêng.

In [10]:
class SolutionMostCommon:
    def minDeletion(self, s, k):
        keep = sum(c for _, c in Counter(s).most_common(k))
        return len(s) - keep

print(SolutionMostCommon().minDeletion("abc", 2))
print(SolutionMostCommon().minDeletion("aabb", 2))
print(SolutionMostCommon().minDeletion("yyyzz", 1))

1
0
2


Độ phức tạp Cách 3: Time `O(n + d log d)`, Space `O(d)`. Ưu điểm: không phải lo `m` âm.

### Cách 4 — Min-heap loại chữ cái ít nhất

Đưa các tần suất vào min-heap; trong khi `len(heap) > k`: `heappop` (loại chữ cái ít nhất) và cộng vào `deleted`. Cho thấy rõ "mỗi lần loại chữ cái rẻ nhất".

In [11]:
import heapq

class SolutionHeap:
    def minDeletion(self, s, k):
        heap = list(Counter(s).values())
        heapq.heapify(heap)
        deleted = 0
        while len(heap) > k:
            deleted += heapq.heappop(heap)
        return deleted

print(SolutionHeap().minDeletion("abc", 2))
print(SolutionHeap().minDeletion("aabb", 2))
print(SolutionHeap().minDeletion("yyyzz", 1))

1
0
2


Độ phức tạp Cách 4: Time `O(n + d log d)`, Space `O(d)`.

## 8. TRACE từng bước

Trace Cách 4 (min-heap) với `s = "abcaab"`, `k = 2` (tần suất `a:3, b:2, c:1`):

| bước | heap (tần suất) | `len(heap) > 2`? | hành động | `deleted` |
|---|---|---|---|---|
| bắt đầu | `[1, 3, 2]` -> heap `[1, 3, 2]` | có (3 > 2) | loại nhỏ nhất `1` (chữ `c`) | 1 |
| sau | `[2, 3]` | không (2 <= 2) | dừng | 1 |

Đáp án `1`.

In [12]:
def trace_heap(s, k):
    heap = list(Counter(s).values())
    heapq.heapify(heap)
    print("tần suất:", dict(Counter(s)), "| heap:", heap)
    deleted = 0
    step = 0
    while len(heap) > k:
        step += 1
        smallest = heapq.heappop(heap)
        deleted += smallest
        print(f"bước {step}: len > {k}, loại chữ cái có {smallest} ký tự | heap={heap} deleted={deleted}")
    print(f"dừng: len(heap)={len(heap)} <= k={k}; đáp án = {deleted}")
    return deleted

trace_heap("abcaab", 2)
trace_heap("yyyzz", 1)
trace_heap("aabb", 2)

tần suất: {'a': 3, 'b': 2, 'c': 1} | heap: [1, 2, 3]
bước 1: len > 2, loại chữ cái có 1 ký tự | heap=[2, 3] deleted=1
dừng: len(heap)=2 <= k=2; đáp án = 1
tần suất: {'y': 3, 'z': 2} | heap: [2, 3]
bước 1: len > 1, loại chữ cái có 2 ký tự | heap=[3] deleted=2
dừng: len(heap)=1 <= k=1; đáp án = 2
tần suất: {'a': 2, 'b': 2} | heap: [2, 2]
dừng: len(heap)=2 <= k=2; đáp án = 0


0

## 9. Code nộp LeetCode

In [13]:
from typing import List
from collections import Counter

class Solution:
    def minDeletion(self, s: str, k: int) -> int:
        freqs = sorted(Counter(s).values())
        m = max(0, len(freqs) - k)
        return sum(freqs[:m])

print(Solution().minDeletion("abc", 2), Solution().minDeletion("aabb", 2), Solution().minDeletion("yyyzz", 1))

1 0 2


## 10. Vì sao thuật toán đúng

- **Mỗi chữ cái: giữ hết hoặc xoá hết.** Số ký tự khác nhau chỉ giảm khi một chữ cái bị xoá **toàn bộ**. Xoá một phần mà không xoá hết chỉ tốn thêm mà không giảm `distinct`. Vậy mọi cách tối ưu tương ứng một tập `S` các chữ cái **giữ lại** với `|S| <= k`, chi phí = `n - sum(cnt[c] for c in S)`.
- **Cần loại đúng `m = max(0, d - k)` chữ cái** (loại nhiều hơn chỉ tốn thêm; loại ít hơn còn `> k`).
- **Tham lam:** chi phí loại tập `T` chữ cái là tổng tần suất của chúng. Với `|T| = m` cố định, tổng nhỏ nhất đạt được khi `T` gồm `m` tần suất **nhỏ nhất** (hoán đổi một phần tử của `T` lấy tần suất lớn hơn bằng một phần tử ngoài `T` nhỏ hơn không làm tổng tăng).
- **Cách 3:** giữ `k` chữ cái nhiều nhất cực đại số ký tự giữ lại, tương đương cực tiểu số xoá.
- **Cách 4:** mỗi `heappop` lấy chữ cái ít nhất còn lại; sau `m` lần heap còn `k` phần tử và tổng đã trừ là tổng `m` tần suất nhỏ nhất.
- **Cách 1** chứng minh bằng vét cạn nên các test đối chiếu xác nhận tham lam đúng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `d` = số chữ cái khác nhau (`d <= 26`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập con chữ cái | `O(n + 2^d * d)` | `O(d)` |
| 2. Sắp xếp tần suất | `O(n + d log d)` | `O(d)` |
| 3. `most_common(k)` | `O(n + d log d)` | `O(d)` |
| 4. Min-heap | `O(n + d log d)` | `O(d)` |

Vì `d <= 26` hằng số nên cách 2, 3, 4 thực chất `O(n)`; thời gian đếm `O(n)` chiếm chủ yếu. Constraints của đề (`n <= 16`) rất nhỏ, nhưng lời giải tham lam vẫn đúng cho `n` lớn.

## 12. Bộ test

Test 1: ví dụ đề và edge case (`n = 1`, `k >= d`, `k = 1`, tất cả chữ cái khác nhau, tất cả giống nhau, `k = 16`).

In [14]:
impls = [SolutionBrute(), SolutionSort(), SolutionMostCommon(), SolutionHeap(), Solution()]
tests = [
    ("abc", 2, 1),
    ("aabb", 2, 0),
    ("yyyzz", 1, 2),
    ("a", 1, 0),                       # n = 1
    ("a", 16, 0),                      # k lớn hơn số chữ cái
    ("aaaa", 1, 0),                    # toàn giống nhau
    ("abcd", 1, 3),                    # mỗi chữ cái một lần, giữ 1
    ("abcd", 4, 0),                    # k = d
    ("abcd", 5, 0),                    # k > d (bắt lỗi slice âm)
    ("aabbbcccc", 2, 2),               # loại a (2)
    ("aabbbcccc", 1, 5),               # giữ c (4), xoá 2 + 3
    ("abcdefghijklmnop", 1, 15),       # 16 chữ cái khác nhau, độ dài 16
    ("abcdefghijklmnop", 16, 0),       # k = 16
    ("zzzzzzzyyyxxw", 2, 3),           # 7 z, 3 y, 2 x, 1 w -> giữ z, y: xoá 2 + 1 = 3
]
for s, k, exp in tests:
    assert 1 <= len(s) <= 16 and 1 <= k <= 16 and s.islower()
    for im in impls:
        got = im.minDeletion(s, k)
        assert got == exp, (type(im).__name__, s, k, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 14


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= len(s) <= 16`, `1 <= k <= 16`, chữ cái thường). Oracle độc lập: thử **mọi tập con của các chữ cái xuất hiện** bằng `itertools.combinations` (chọn `min(k, d)` chữ cái giữ lại) — viết khác cách mặt nạ bit.

In [15]:
import random
from itertools import combinations
random.seed(3545)

def oracle(s, k):
    letters = sorted(set(s))
    r = min(k, len(letters))
    best_keep = 0
    for chosen in combinations(letters, r):
        keep = sum(1 for ch in s if ch in chosen)
        best_keep = max(best_keep, keep)
    return len(s) - best_keep

for t in range(2000):
    n = random.randint(1, 16)
    alphabet = random.choice(["ab", "abc", "abcdef", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(random.choice(alphabet) for _ in range(n))
    k = random.randint(1, 16)
    assert 1 <= len(s) <= 16 and 1 <= k <= 16
    exp = oracle(s, k)
    for im in impls:
        assert im.minDeletion(s, k) == exp, (type(im).__name__, s, k)
print("2000 test ngẫu nhiên OK")

2000 test ngẫu nhiên OK


Test 3: tính chất — đáp án `0` khi `k >= số chữ cái khác nhau`; `0 <= đáp án < len(s)`; đáp án không **tăng** khi tăng `k`; chuỗi sau khi xoá (giữ `k` chữ cái nhiều nhất) có `<= k` chữ cái khác nhau và độ dài `len(s) - đáp án`.

In [16]:
random.seed(45)
for _ in range(1000):
    n = random.randint(1, 16)
    s = "".join(random.choice("abcdefgh") for _ in range(n))
    k = random.randint(1, 10)
    ans = Solution().minDeletion(s, k)
    d = len(set(s))
    assert 0 <= ans < len(s)
    if k >= d:
        assert ans == 0
    assert Solution().minDeletion(s, k + 1) <= ans
    kept_letters = {ch for ch, _ in Counter(s).most_common(k)}
    remaining = [ch for ch in s if ch in kept_letters]
    assert len(set(remaining)) <= k and len(remaining) == len(s) - ans
print("tính chất (0 khi k >= d, không tăng theo k, chuỗi còn lại hợp lệ): OK")

tính chất (0 khi k >= d, không tăng theo k, chuỗi còn lại hợp lệ): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: xoá chữ cái **nhiều nhất** thay vì ít nhất

In [17]:
def wrong_remove_most(s, k):
    freqs = sorted(Counter(s).values(), reverse=True)    # SAI: giảm dần
    m = max(0, len(freqs) - k)
    return sum(freqs[:m])

s, k = "yyyzz", 1
print("sai :", wrong_remove_most(s, k))
print("đúng:", Solution().minDeletion(s, k))
assert wrong_remove_most(s, k) != Solution().minDeletion(s, k)

sai : 3
đúng: 2


Vì sao sai: sắp xếp giảm dần rồi cộng đầu danh sách là xoá `y` (3 ký tự) thay vì `z` (2 ký tự). Muốn xoá **ít** nhất thì loại các chữ cái **hiếm** nhất. **Sửa:** sắp xếp **tăng dần**.

### Lỗi 2: trả về số **chữ cái** cần loại thay vì số **ký tự** bị xoá

In [18]:
def wrong_count_letters(s, k):
    return max(0, len(set(s)) - k)              # SAI: đếm số chữ cái, không phải số ký tự

s, k = "yyyzz", 1
print("sai :", wrong_count_letters(s, k))
print("đúng:", Solution().minDeletion(s, k))
assert wrong_count_letters(s, k) != Solution().minDeletion(s, k)

sai : 1
đúng: 2


Vì sao sai: để loại chữ `z` phải xoá **cả hai** ký tự `z`, tức `2` lần xoá, không phải `1`. Đề hỏi **số ký tự** bị xoá. **Sửa:** cộng tần suất của các chữ cái bị loại.

### Lỗi 3: slice với `m` âm khi `k > d`

In [19]:
def wrong_negative_slice(s, k):
    freqs = sorted(Counter(s).values())
    return sum(freqs[:len(freqs) - k])          # SAI: len - k âm khi k > d -> cắt kiểu "bỏ phần tử cuối"

s, k = "aabb", 3
print("sai :", wrong_negative_slice(s, k))
print("đúng:", Solution().minDeletion(s, k))
assert wrong_negative_slice(s, k) != Solution().minDeletion(s, k)

sai : 2
đúng: 0


Vì sao sai: `d = 2`, `k = 3` nên `len - k = -1`, `freqs[:-1]` lấy tất cả trừ phần tử cuối (`[2]`) -> tổng `2`, trong khi chuỗi đã thoả (đáp án `0`). **Sửa:** `m = max(0, d - k)`, hoặc dùng cách `most_common(k)`.

### Lỗi 4: xoá từng ký tự lẻ tẻ (không xoá hết một chữ cái)

In [20]:
def wrong_partial_delete(s, k):
    d = len(set(s))
    return max(0, d - k)                        # SAI: giả định mỗi chữ cái chỉ cần xoá 1 ký tự

s, k = "yyyzz", 1
print("sai :", wrong_partial_delete(s, k))
print("đúng:", Solution().minDeletion(s, k))
assert wrong_partial_delete(s, k) != Solution().minDeletion(s, k)

sai : 1
đúng: 2


Vì sao sai: `d - k = 1` nhưng để biến `"yyyzz"` thành chuỗi có 1 chữ cái phải xoá **hết** `z` (2 ký tự). Mỗi chữ cái bị loại tốn **tần suất của nó**, không phải `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa `k` ký tự khác nhau" + "xoá ít nhất" -> **giữ `k` chữ cái nhiều nhất**.
- Hễ gặp "xoá để còn ít loại hơn" thì mỗi loại **xoá trọn** (chi phí = tần suất), không xoá lẻ.
- Đếm tần suất bằng `Counter` (hoặc `[0] * 26`) rồi chỉ làm việc trên danh sách tần suất — **bỏ qua** vị trí ký tự.
- Đề có `n <= 16` gợi ý cả **vét cạn bitmask**; nhưng tham lam trên tần suất luôn đủ và đúng.
- Cẩn thận slice với chỉ số âm: dùng `max(0, d - k)` hoặc `most_common(k)`.
- Kiểm tra: `k >= d`, `k = 1`, mọi ký tự khác nhau, mọi ký tự giống nhau.

## 15. Biến thể của bài

1. **Xoá để mọi ký tự có tần suất duy nhất** (`1647. Minimum Deletions to Make Character Frequencies Unique`).
2. **Chuỗi con dài nhất có tối đa `k` ký tự khác nhau** (`340`, cửa sổ trượt) — không phải xoá mà là chọn đoạn liên tiếp.
3. **Mỗi ký tự có chi phí xoá riêng** (chi phí = tần suất × giá): sắp xếp theo tổng chi phí.
4. Yêu cầu trả về **chuỗi còn lại** (hoặc tập chữ cái được giữ), không chỉ số lượng.
5. Phiên bản **`n` lớn** (`10^5`): cách tham lam y nguyên, chỉ khác là không thể vét cạn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập chữ cái | thử mọi tập `<= k` chữ cái giữ lại | `O(n + 2^d d)` | `O(d)` | chuẩn chân lý, `d` nhỏ |
| 2. Sắp xếp tần suất | cộng `d - k` tần suất nhỏ nhất | `O(n + d log d)` | `O(d)` | **chuẩn, nên nộp** |
| 3. `most_common(k)` | giữ `k` chữ cái nhiều nhất | `O(n + d log d)` | `O(d)` | không lo slice âm |
| 4. Min-heap | loại chữ cái ít nhất cho tới khi còn `k` | `O(n + d log d)` | `O(d)` | thể hiện rõ tham lam |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minDeletion("aabbbcd", 2)` bằng bao nhiêu?

**Bài 2 (tính tay):** `s = "xxyyyzzzzw"`, `k = 2`: tần suất từng chữ cái, những chữ nào bị loại, xoá bao nhiêu ký tự?

**Bài 3 (điền TODO):** hoàn thành `letters_to_remove(freqs, k)` trả về số chữ cái cần loại, `max(0, len(freqs) - k)`.

**Bài 4 (tự viết lại):** viết `kept_letters(s, k)` trả về **tập** các chữ cái được giữ (dùng `most_common`; nếu hoà thì ưu tiên chữ cái nhỏ theo thứ tự bảng chữ cái).

**Bài 5 (mini challenge):** viết `min_cost_delete(s, k, price)` với `price[c]` là chi phí xoá mỗi ký tự `c` (dict); trả về tổng chi phí nhỏ nhất (loại các chữ cái có tổng `cnt * price` nhỏ nhất).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minDeletion("aabbbcd", 2))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def letters_to_remove(freqs, k):
    # TODO: số chữ cái phải loại để còn tối đa k (không âm)
    return -1

2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** đếm tần suất mỗi chữ cái; giữ những chữ cái nhiều nhất, xoá những chữ cái ít nhất.

**Mức 2 (vừa):** Bài 3: `max(0, len(freqs) - k)`. Bài 4: `sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:k]`. Bài 5: tính `cost[c] = cnt[c] * price[c]` cho mỗi chữ cái, sắp xếp tăng dần, cộng `d - k` giá trị đầu.

**Mức 3 (gần đáp án):** Bài 1: tần suất `a:2, b:3, c:1, d:1`, `d = 4`, loại `2` chữ cái nhỏ nhất (`c`, `d`) -> `2`. Bài 2: `x:2, y:3, z:4, w:1`, loại `w` và `x` -> `1 + 2 = 3`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: a:2 b:3 c:1 d:1 -> loại c, d -> 2
assert Solution().minDeletion("aabbbcd", 2) == 2

# Bài 2: x:2 y:3 z:4 w:1 -> loại w (1) và x (2) -> 3
assert Solution().minDeletion("xxyyyzzzzw", 2) == 3

# Bài 3
def letters_to_remove(freqs, k):
    return max(0, len(freqs) - k)
assert letters_to_remove([3, 2, 1], 2) == 1
assert letters_to_remove([3, 2], 5) == 0
assert letters_to_remove([1, 1, 1, 1], 1) == 3

# Bài 4
def kept_letters(s, k):
    cnt = Counter(s)
    top = sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:k]
    return {ch for ch, _ in top}
assert kept_letters("yyyzz", 1) == {"y"}
assert kept_letters("abcaab", 2) == {"a", "b"}
assert kept_letters("ab", 5) == {"a", "b"}

# Bài 5
def min_cost_delete(s, k, price):
    cnt = Counter(s)
    costs = sorted(cnt[c] * price[c] for c in cnt)
    m = max(0, len(costs) - k)
    return sum(costs[:m])
unit = {c: 1 for c in "abcdefghijklmnopqrstuvwxyz"}
assert min_cost_delete("yyyzz", 1, unit) == Solution().minDeletion("yyyzz", 1)
assert min_cost_delete("abcaab", 2, unit) == 1
assert min_cost_delete("yyyzz", 1, {**unit, "z": 10}) == 3      # lúc này loại y (3 * 1) rẻ hơn loại z (2 * 10)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "giữ `k` loại nhiều nhất, xoá trọn các loại còn lại":**

```python
freqs = sorted(Counter(s).values())      # tăng dần
m = max(0, len(freqs) - k)               # số chữ cái phải loại
return sum(freqs[:m])                    # xoá trọn m chữ cái hiếm nhất
```

**Checklist trước khi submit:**

- [ ] Chữ cái bị loại thì xoá **trọn** (chi phí = tần suất, không phải `1`)?
- [ ] Sắp xếp tần suất **tăng dần** (loại chữ hiếm trước)?
- [ ] `m = max(0, d - k)` (không để chỉ số âm khi `k >= d`)?
- [ ] Đáp án là **số ký tự** bị xoá (không phải số chữ cái)?
- [ ] Đã thử `k >= d`, `k = 1`, `n = 1`, mọi ký tự khác nhau?
- [ ] Chữ ký đúng `minDeletion(self, s: str, k: int) -> int`?